# 09A – Model Serialization & Production Packaging

Prepare the finalized bankruptcy prediction model for production deployment.

## Business Objective
Package the trained model with metadata, verify it can be reloaded, and create deployment-ready artifacts.

In [1]:
import joblib
import json
import hashlib
from datetime import datetime
from pathlib import Path
import pandas as pd

MODEL_PATH='../models/production_bankruptcy_model.joblib'
DATA_PATH='../data/datasets/american_bankruptcy_cleaned.csv'


In [2]:
# Load model and dataset
model=joblib.load(MODEL_PATH)
df=pd.read_csv(DATA_PATH)

target='status_label' if 'status_label' in df.columns else 'target'
feature_names=[c for c in df.columns if c not in (target,'company_name')]

print(type(model))
print(f'Features: {len(feature_names)}')


<class 'sklearn.ensemble._forest.RandomForestClassifier'>
Features: 19


In [3]:
# Verify model inference
sample=df[feature_names].head(5)
pred=model.predict(sample)
prob=model.predict_proba(sample)

verification=pd.DataFrame({
    'Prediction':pred,
    'Probability':prob.max(axis=1)
})
verification.to_csv('model_verification_predictions.csv',index=False)
verification


,Prediction,Probability
0,0,0.928235
1,0,0.905646
2,0,0.740655
3,0,0.726394
4,0,0.808414


In [4]:
# Create metadata
sha256=hashlib.sha256(Path(MODEL_PATH).read_bytes()).hexdigest()

metadata={
    "model_name":"Bankruptcy Risk Prediction",
    "model_file":MODEL_PATH,
    "algorithm":type(model).__name__,
    "version":"1.0.0",
    "dataset":"../data/datasets/american_bankruptcy_cleaned.csv",
    "feature_count":len(feature_names),
    "created_utc":datetime.utcnow().isoformat()+"Z",
    "sha256":sha256
}

with open("../models/model_metadata.json","w") as f:
    json.dump(metadata,f,indent=4)

pd.DataFrame({
    "Feature":feature_names
}).to_csv("model_schema.csv",index=False)

metadata


/tmp/ipykernel_1945/1093750589.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_utc":datetime.utcnow().isoformat()+"Z",


{'model_name': 'Bankruptcy Risk Prediction',
 'model_file': '../models/production_bankruptcy_model.joblib',
 'algorithm': 'RandomForestClassifier',
 'version': '1.0.0',
 'dataset': '../data/datasets/american_bankruptcy_cleaned.csv',
 'feature_count': 19,
 'created_utc': '2026-08-08T11:02:08.019928Z',
 'sha256': '1fca8cfdb755bdd267f749df068bc76225d1bff159de6b00bf222b0c8b8f2bcc'}

## Production Checklist

- ✔ Model successfully loads
- ✔ Prediction verified
- ✔ Feature schema exported
- ✔ Model metadata generated
- ✔ Ready for API, Streamlit, and Docker deployment

## Deliverables

- `model_metadata.json`
- `model_schema.csv`
- `model_verification_predictions.csv`

These artifacts are used by the remaining deployment notebooks.